# KORMo 1B — from-scratch 사전학습 (Colab A100 전용)

[MLP-Lab/KORMo-tutorial](https://github.com/MLP-Lab/KORMo-tutorial)의 `01.pretrain_from_scratch.ipynb`를 Colab A100에서 바로 실행되도록 조정한 버전입니다.

**사용법**: 런타임 → 런타임 유형 변경 → **A100 GPU** 선택 → 저장 → 런타임 → 모두 실행

원본 대비 변경점:
- uv 가상환경·flash-attn 빌드 제거 (본 노트북은 PyTorch 내장 `flex_attention` 사용 → flash-attn 불필요)
- `num_proc`을 Colab vCPU 수에 맞게 자동 설정 (원본: 48/128 하드코딩)
- Google Drive 저장 + **모드 자동 분기**: 새 학습(fresh) / 끊긴 학습 재개(resume) / 학습된 모델 위에 추가 학습(continue·continue-prev)
- **검증셋 분리 + 학습 중 평가**: 고정 seed로 128 시퀀스를 떼어 주기적으로 `eval/loss` 로깅 — multi-epoch 암기 착시와 무관한 진짜 진행 지표
- **데이터 소스 선택** (`DATA_SOURCE`): 튜토리얼 소형 코퍼스(~10M 토큰) 또는 **fineweb-2 한국어**를 원하는 토큰 수만큼 스트리밍 (기본: 1B 토큰)
- **데이터·모드별 학습률 분기**: fresh 5e-4 / 튜토리얼 continue 5e-5 / fineweb2 이어학습 1e-4+warmup

예상 소요 (A100 40GB 기준):
- 튜토리얼 데이터 3 epoch: 데이터 준비 ~10분 + 학습 1~2시간
- **fineweb2 1B 토큰: 데이터 준비 ~1시간(최초 1회, Drive 캐시) + 학습 ~17시간** — 세션이 끊기면 같은 노트북을 다시 실행하세요. 체크포인트에서 자동 재개됩니다

In [ ]:
# GPU 확인 — A100이 보여야 합니다
!nvidia-smi

## 0. 환경 준비

Colab에 torch·accelerate·pyyaml은 이미 있으므로 transformers 버전 고정과 datasets만 설치합니다.
(원본 리포가 고정한 4.57.0은 yank된 버전이라 패치 버전 4.57.1 사용)

In [ ]:
import os
# CUDA 메모리 단편화 완화 — vocab 125k 모델은 loss 계산 시 수 GiB 단위 logits 할당이
# 반복되므로 단편화에 취약 (torch import 전에 설정해야 적용됨)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# HF 다운로드 경로 — 토큰 유무에 따라 분기 (huggingface_hub import 전에 결정해야 함):
# - HF_TOKEN 있음: Xet 네이티브 프로토콜 사용. 브리지 CDN(us.gcp.cdn.hf.co)을 거치지 않아
#   Colab(GCP 리전)에서 발생하는 403 SignatureError(invalid key pair id) 장애를 우회
# - HF_TOKEN 없음: Xet 비활성 → 일반 HTTP 폴백 (익명 Xet 접근은 cas-server 401이 나므로)
FORCE_DISABLE_XET = False   # 토큰이 정상인데도 다운로드가 계속 실패하면 True로 바꿔 일반 HTTP로 강제 폴백

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN 로드됨 — Xet 네이티브 경로 사용")
except Exception:
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    print("HF_TOKEN 없음 — Xet 비활성 (브리지 CDN 폴백)")

if FORCE_DISABLE_XET:
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    print("FORCE_DISABLE_XET — Xet 비활성 (일반 HTTP 폴백)")

!git clone https://github.com/MLP-Lab/KORMo-tutorial.git /content/KORMo-tutorial
%pip install -q "transformers==4.57.1" "datasets>=4.1.1" wandb hf_xet

import sys
sys.path.append('/content/KORMo-tutorial/src')
NUM_PROC = os.cpu_count()
print(f"vCPU: {NUM_PROC}")

### Google Drive 저장 설정 + 데이터 소스 선택

`USE_DRIVE = True`(기본값)면 학습 결과가 Drive에 남아 **다음 세션에서 자동으로 이어집니다**.
`DATA_SOURCE`로 학습 데이터를 고릅니다 — 데이터 소스별로 output 디렉토리(스테이지)가 분리됩니다:

```
MyDrive/kormo-1B-PT/
├── output/               # 스테이지 1: 튜토리얼 코퍼스 학습 (checkpoint-*, final/)
├── output-fw2-1000M/     # 스테이지 2: fineweb2 1B 토큰 이어학습 (checkpoint-*, final/)
├── data/                 # fineweb2 토크나이즈+패킹 캐시 (1B 토큰 ≈ 4GB)
└── evals/                # 평가 이력 (02 노트북이 기록)
```

현재 스테이지의 output 상태에 따라 모드가 자동 분기됩니다:

| 스테이지 output 상태 | 동작 (자동 분기) |
|---|---|
| `final/` 있음 (이 스테이지 학습 완료) | 그 가중치 위에 추가 학습 (**continue**) |
| `checkpoint-*`만 있음 (학습 중 끊김) | optimizer 상태까지 복원해 정확히 재개 (**resume**) |
| 비어 있고 이전 스테이지 `output/final` 있음 (fineweb2만) | 이전 완성본에서 시작 (**continue-prev**) |
| 아무것도 없음 | 새 모델 초기화 후 학습 (**fresh**) |

`final/`보다 `checkpoint-*`를 먼저 확인하지 않는 이유: 학습 완주 후 남은 낡은 체크포인트를
재개하는 사고를 막기 위해서입니다. 스테이지 분리 덕분에 fineweb2 장기 학습이 끊겨도
이전 스테이지의 `final/` 때문에 resume이 무시되는 일이 없습니다.

**처음부터 다시 학습하려면** `FORCE_FRESH = True` — 현재 스테이지 output을
백업 폴더로 옮긴 뒤 처음부터 시작합니다.

용량 (fineweb2 1B 기준): 데이터 캐시 ~4GB + 스테이지별 체크포인트 ~8GB + final ~2.6GB ≈ **15GB 이상 필요**.
무료 15GB로는 빠듯하므로 이전 스테이지 체크포인트(`output/checkpoint-*`)를 미리 정리하는 것을 권합니다.

In [ ]:
import os, glob, shutil
from datetime import datetime

USE_DRIVE = True
FORCE_FRESH = False   # True면 기존 결과를 백업 폴더로 옮기고 처음부터 재학습

# ── 학습 데이터 선택 ──────────────────────────────────────────────────────
# 'tutorial': KORMo 튜토리얼 소형 코퍼스 (~10M 토큰, 3 epoch 반복)
# 'fineweb2': fineweb-2 한국어(kor_Hang) 웹 코퍼스를 TARGET_TOKENS만큼 스트리밍 (1 epoch)
DATA_SOURCE = 'fineweb2'
TARGET_TOKENS = 1_000_000_000   # fineweb2에서 가져올 학습 토큰 수
SEQ_LEN = 4096                  # 패킹 시퀀스 길이 (데이터 캐시와 collator가 공유)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/kormo-1B-PT'
else:
    BASE_DIR = '/content/kormo-1B-PT'

# 구버전 레이아웃 마이그레이션 — 예전엔 final/·checkpoint-*를 BASE_DIR 바로 아래 저장했음.
# output/으로 옮겨두지 않으면 기존 학습 결과를 못 찾고 fresh 모드로 잘못 시작함
TUT_OUTPUT = os.path.join(BASE_DIR, 'output')
if not os.path.isdir(TUT_OUTPUT):
    old = glob.glob(os.path.join(BASE_DIR, 'checkpoint-*'))
    if os.path.isdir(os.path.join(BASE_DIR, 'final')):
        old.append(os.path.join(BASE_DIR, 'final'))
    if old:
        os.makedirs(TUT_OUTPUT)
        for p in old:
            shutil.move(p, TUT_OUTPUT)
            print(f"구버전 레이아웃 이동: {p} → {TUT_OUTPUT}/")

# 데이터 소스별로 output을 분리(스테이지) — fineweb2 장기 학습이 도중에 끊겨도
# 튜토리얼 학습의 final/과 섞이지 않아 resume 판정이 정확하게 동작함
STAGE_NAME = f"output-fw2-{TARGET_TOKENS // 1_000_000}M" if DATA_SOURCE == 'fineweb2' else 'output'
OUTPUT_DIR = os.path.join(BASE_DIR, STAGE_NAME)
PREV_FINAL = os.path.join(TUT_OUTPUT, 'final')   # 이전 스테이지(튜토리얼 학습) 완성본

if FORCE_FRESH and os.path.isdir(OUTPUT_DIR):
    backup = os.path.join(BASE_DIR, f"{STAGE_NAME}-backup-{datetime.now():%Y%m%d-%H%M%S}")
    shutil.move(OUTPUT_DIR, backup)
    print(f"기존 결과 백업됨: {backup}")

FINAL_DIR = os.path.join(OUTPUT_DIR, 'final')

def latest_checkpoint(d):
    cks = glob.glob(os.path.join(d, 'checkpoint-*'))
    return max(cks, key=lambda p: int(p.rsplit('-', 1)[-1])) if cks else None

if os.path.isdir(FINAL_DIR):
    MODE = 'continue'        # 이 스테이지의 완료본 위에 추가 학습
elif latest_checkpoint(OUTPUT_DIR):
    MODE = 'resume'          # 이 스테이지의 끊긴 학습을 체크포인트에서 재개
elif DATA_SOURCE == 'fineweb2' and os.path.isdir(PREV_FINAL):
    MODE = 'continue-prev'   # 튜토리얼 학습 완성본을 시작 가중치로 새 스테이지 시작
else:
    MODE = 'fresh'           # 무작위 초기화로 처음부터 학습

print(f"데이터: {DATA_SOURCE} | 저장 위치: {OUTPUT_DIR}\n실행 모드: {MODE}")

## 1. 데이터 준비 — Raw text → Tokenize → Sequence packing

토크나이저는 `KORMo-Team/KORMo-tokenizer`(vocab 125,000)를 사용합니다.
`tokenizer.encode()`가 문서 앞에 `<|BOS|>`를 자동으로 붙이는데, 이 BOS 위치가 뒤에서 문서 경계 마스크의 기준이 됩니다.

- **tutorial**: 소형 코퍼스(16K 문서, ~10M 토큰) 전체를 다운로드 후 `map`으로 토크나이즈·패킹
- **fineweb2**: `HuggingFaceFW/fineweb-2`의 한국어(`kor_Hang`, 전체 ~106GB) 서브셋을 **스트리밍**으로
  `TARGET_TOKENS`만큼만 읽으면서 토크나이즈·패킹 → 결과를 Drive에 캐시.
  최초 1회만 ~1시간이 걸리고(실측 ~300K 토큰/초), 이후 세션은 캐시를 바로 로드합니다.
  `keep_doc()` 함수로 문서 품질 필터를 직접 정할 수 있습니다 — kor_Hang에도 영어 위주 문서가
  일부 섞여 있어(`language_score` 낮음) 필터를 정해볼 가치가 있습니다.

In [ ]:
import datasets, time
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('KORMo-Team/KORMo-tokenizer')
print("vocab:", tokenizer.vocab_size, "| bos:", tokenizer.bos_token, "| eos:", tokenizer.eos_token)

# HF CDN 순단 대비 재시도 (다운로드는 캐시되므로 재시도는 이어받기가 됨)
def retry_wait(e, attempt, max_attempts=5):
    """에러 내용을 그대로 출력하고 상태 코드에 맞게 대기/중단 결정."""
    code = getattr(getattr(e, 'response', None), 'status_code', None)
    print(f"[{attempt}/{max_attempts}] 실패: {type(e).__name__}(HTTP {code}) — {str(e)[:300]}")
    if code == 401:
        raise RuntimeError("401 인증 실패 — Colab 보안 비밀 HF_TOKEN이 만료/무효입니다. 재발급 후 다시 실행하세요") from e
    wait = 90 * attempt if code == 429 else 30   # 429(rate limit)는 길게 기다려야 풀림
    print(f"  → {wait}초 후 재시도")
    time.sleep(wait)

In [ ]:
import numpy as np

def keep_doc(doc):
    """fineweb2 문서 필터 — True를 반환한 문서만 학습에 사용.

    사용 가능한 필드: doc['text'], doc['language_score'](0~1, 한국어 확신도),
    doc['url'], doc['date'], doc['minhash_cluster_size'](중복 클러스터 크기) 등.

    TODO(직접 결정): 품질 기준을 정하세요. fineweb-2는 이미 기본 품질 필터를
    거쳤으므로 그대로(True) 써도 되지만, 예를 들어
      - 너무 짧은 문서 제외:  len(doc['text']) >= 200
      - 언어 확신도 하한:     doc['language_score'] >= 0.9
    같은 기준을 추가하면 같은 토큰 예산으로 더 좋은 데이터를 담을 수 있습니다.
    """
    return True

def build_fineweb2_packed():
    """fineweb-2 kor_Hang을 스트리밍으로 받아 TARGET_TOKENS만큼 토크나이즈+패킹.

    전체(~106GB)를 받지 않고 필요한 만큼만 앞에서부터 읽는다. 문서 경계에는
    튜토리얼 파이프라인과 동일하게 BOS(encode가 자동 추가)·EOS를 넣어
    intra-document mask가 그대로 동작한다.
    """
    stream = datasets.load_dataset('HuggingFaceFW/fineweb-2', name='kor_Hang',
                                   split='train', streaming=True)
    target_seqs = TARGET_TOKENS // SEQ_LEN
    seqs = []                             # (SEQ_LEN,) int32 배열 목록
    carry = np.empty(0, dtype=np.int32)   # 시퀀스 경계에 걸친 나머지 토큰
    texts, n_docs = [], 0
    next_report, t0 = 50_000_000, time.time()

    def flush(texts, carry):
        ids = tokenizer(texts)['input_ids']   # fast tokenizer 배치 인코딩 (멀티스레드)
        flat = np.concatenate(
            [carry] + [np.asarray(x + [tokenizer.eos_token_id], dtype=np.int32) for x in ids])
        n = len(flat) // SEQ_LEN
        seqs.extend(flat[:n * SEQ_LEN].reshape(n, SEQ_LEN))
        return flat[n * SEQ_LEN:]

    for doc in stream:
        if not keep_doc(doc):
            continue
        texts.append(doc['text'])
        n_docs += 1
        if len(texts) >= 1024:
            carry = flush(texts, carry)
            texts = []
            if len(seqs) >= target_seqs:
                break
            if len(seqs) * SEQ_LEN >= next_report:
                print(f"  {n_docs:,} 문서 → {len(seqs)*SEQ_LEN/1e6:.0f}M/{TARGET_TOKENS/1e6:.0f}M 토큰"
                      f" ({time.time()-t0:.0f}초 경과)")
                next_report += 50_000_000
    assert len(seqs) >= target_seqs, "스트림이 먼저 소진됨 — TARGET_TOKENS가 (필터 통과) 코퍼스보다 큽니다"
    return datasets.Dataset.from_dict({'input_ids': np.stack(seqs[:target_seqs])})

if DATA_SOURCE == 'fineweb2':
    # 토크나이즈+패킹 결과를 Drive에 캐시 (int32, 1B 토큰 ≈ 4GB)
    # → 세션이 끊겨도 다음 세션은 스트리밍 없이 캐시만 로드
    CACHE_DIR = os.path.join(BASE_DIR, 'data', f"fw2-kor-{TARGET_TOKENS // 1_000_000}M-seq{SEQ_LEN}")
    if os.path.isdir(CACHE_DIR):
        packed_ds = datasets.load_from_disk(CACHE_DIR)
        print(f"캐시 로드: {CACHE_DIR} — {len(packed_ds):,} 시퀀스")
    else:
        packed_ds = None
        for attempt in range(1, 6):
            try:
                packed_ds = build_fineweb2_packed()
                break
            except Exception as e:
                retry_wait(e, attempt)   # 스트리밍 실패는 처음부터 다시 (캐시 저장 전 1회성 비용)
        assert packed_ds is not None, "5회 재시도 실패 — 위 에러 메시지 확인"
        packed_ds.save_to_disk(CACHE_DIR)
        print(f"캐시 저장: {CACHE_DIR}")
else:
    # 튜토리얼 소형 코퍼스 — 기존 파이프라인 (load → shuffle → tokenize → pack)
    from itertools import chain

    pt_dataset = None
    for attempt in range(1, 6):
        try:
            pt_dataset = datasets.load_dataset(
                'KORMo-Team/KORMo-tutorial-datasets', name='pretrain', split='train')
            break
        except Exception as e:
            retry_wait(e, attempt)
    assert pt_dataset is not None, ("5회 재시도 실패 — 위 에러 메시지 확인: 403이면 환경 셀의 FORCE_DISABLE_XET=True로 재시도, "
                                    "429면 시간을 두고 재시도, 5xx면 status.huggingface.co 확인")
    train_ds = pt_dataset.shuffle(seed=42)

    def _tokenize(examples, tokenizer):
        return {'input_ids': [tokenizer.encode(t) + [tokenizer.eos_token_id] for t in examples['text']]}

    def _pack_dataset(examples, seq_len):
        flat = list(chain.from_iterable(examples['input_ids']))
        n_full = len(flat) // seq_len
        return {'input_ids': [flat[i*seq_len:(i+1)*seq_len] for i in range(n_full)]}

    tokenized_ds = train_ds.map(_tokenize, batched=True, num_proc=NUM_PROC,
                                remove_columns=train_ds.column_names, fn_kwargs={'tokenizer': tokenizer})
    packed_ds = tokenized_ds.map(_pack_dataset, batched=True, batch_size=100_000, num_proc=NUM_PROC,
                                 remove_columns=tokenized_ds.column_names, fn_kwargs={'seq_len': SEQ_LEN})

In [ ]:
# 검증셋 분리 — 데이터가 결정적(fineweb2: 고정 캐시 / tutorial: seed 고정 파이프라인)이고
# split도 seed=42라 매 세션 정확히 같은 시퀀스가 분리됨 → 검증 데이터가 학습에 새지 않음
packed_ds.set_format('torch')
split = packed_ds.train_test_split(test_size=128, seed=42)
packed_ds, val_ds = split['train'], split['test']

total_tokens = len(packed_ds) * SEQ_LEN
print(packed_ds)
print(f"학습 토큰: {total_tokens/1e6:.1f}M | 검증: {len(val_ds)} 시퀀스 ({len(val_ds)*SEQ_LEN/1e6:.1f}M 토큰)")

## 2. Intra-document attention mask (flex_attention)

패킹으로 한 시퀀스에 여러 문서가 섞이므로, BOS 토큰의 누적합으로 문서 ID를 만들어
**같은 문서 안에서만 attention이 가도록** block mask를 만듭니다. causal mask와 AND로 결합합니다.

In [ ]:
import torch
from torch.nn.attention.flex_attention import create_block_mask, and_masks

def _intra_doc_mask(input_ids, bos_token_id):
    is_bos = (input_ids == bos_token_id)
    doc_ids = torch.cumsum(is_bos.flatten().long(), 0).view_as(input_ids)

    def intra_doc(b, h, q_idx, kv_idx):
        return doc_ids[b, q_idx] == doc_ids[b, kv_idx]

    def causal(b, h, q_idx, kv_idx):
        return q_idx >= kv_idx

    return and_masks(intra_doc, causal)

def create_intra_doc_mask(input_ids, tokenizer):
    B, Q_LEN = input_ids.shape
    mask_mod = _intra_doc_mask(input_ids.to('cuda'), tokenizer.bos_token_id)
    return create_block_mask(mask_mod=mask_mod, B=B, H=None, Q_LEN=Q_LEN, KV_LEN=Q_LEN)

In [ ]:
from dataclasses import dataclass
from torch.nn.utils.rnn import pad_sequence
from transformers import PreTrainedTokenizer

@dataclass
class DataCollatorIntraDocMask:
    tokenizer: PreTrainedTokenizer

    def __call__(self, instances):
        input_ids = [inst["input_ids"][:SEQ_LEN] for inst in instances]
        input_ids = pad_sequence(input_ids, batch_first=True,
                                 padding_value=self.tokenizer.pad_token_id)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        block_mask = create_intra_doc_mask(input_ids, self.tokenizer)
        return dict(input_ids=input_ids, labels=labels, attention_mask=block_mask)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 3. 모델 준비 — 모드에 따라 초기화 / 로드

1B config: hidden 2048 / 16 layers / GQA(16 heads, 8 KV) / vocab 125,184 → 약 1.3B 파라미터, bf16.
A100 40GB에서 batch 4 × 4096 토큰이 여유 있게 들어갑니다.

- **fresh / resume**: 무작위 초기화 (resume이면 학습 셀에서 체크포인트 가중치로 덮어써짐)
- **continue**: 현재 스테이지의 `final/`에서 학습된 가중치 로드
- **continue-prev**: 이전 스테이지(튜토리얼 학습)의 `output/final`에서 가중치 로드 — fineweb2 이어학습의 시작점

KORMo는 커스텀 아키텍처라 `KORMoForCausalLM`으로 직접 로드합니다.

In [ ]:
import torch
from kormo.train.arguments import KORMoTrainingArguments
from kormo.train.trainer import KORMoTrainer
from kormo.modeling_configs.load_model import load_model_from_config
from kormo.model._modeling_kormo import KORMoForCausalLM

if MODE == 'continue':
    model = KORMoForCausalLM.from_pretrained(
        FINAL_DIR, dtype=torch.bfloat16,
        _attn_implementation='flex_attention',
    )
elif MODE == 'continue-prev':
    # 새 스테이지의 시작 가중치 = 이전 스테이지(튜토리얼 학습) 완성본
    model = KORMoForCausalLM.from_pretrained(
        PREV_FINAL, dtype=torch.bfloat16,
        _attn_implementation='flex_attention',
    )
    print("이전 스테이지 가중치에서 시작:", PREV_FINAL)
else:  # fresh, resume
    model, _ = load_model_from_config('1B', _attn_implementation='flex_attention')

model.to('cuda')
n_params = sum(p.numel() for p in model.parameters())
print(f"[{MODE}] 파라미터: {n_params/1e9:.2f}B | attn: {model.config._attn_implementation} | dtype: {next(model.parameters()).dtype}")

## 4. wandb 로깅 설정

학습 상태(loss·학습률·토큰 처리량·GPU 사용률)를 [wandb.ai](https://wandb.ai)에서 실시간으로 관찰합니다.
Colab 세션이 끊겨도 로그는 wandb에 남습니다.

**최초 1회 설정** (API 키를 노트북에 넣지 않는 안전한 방식):
1. https://wandb.ai/authorize 에서 API 키 복사
2. Colab 좌측 사이드바 **🔑 (보안 비밀)** → `WANDB_API_KEY` 이름으로 등록 → "노트북 액세스" 켜기

보안 비밀이 없으면 셀 실행 시 키 입력 프롬프트가 뜹니다. `USE_WANDB = False`로 끌 수도 있습니다.

**세션이 끊겨도 하나의 run으로**: wandb run의 정체성은 이름이 아니라 **ID**입니다.
run ID를 스테이지 output 디렉토리에 저장해두고, **resume 모드**에서는 같은 ID로 이어서
기록합니다(`WANDB_RUN_ID` + `WANDB_RESUME=allow`) — 끊긴 학습의 곡선이 한 run에 이어집니다.
마지막 체크포인트 이후에 이미 기록됐던 구간(최대 저장 주기만큼)은 재개 후 다시 지나가면서
"step must be monotonically increasing" 경고와 함께 건너뛰는데, 정상입니다.
fresh/continue/continue-prev는 `global_step`이 0부터 시작하는 새 학습이므로 새 run을 만듭니다
(기존 run에 붙이면 이미 지나간 스텝이라 로그가 전부 거부됨).

**상태 표기**: wandb는 `wandb.finish()` 없이 세션이 죽으면 학습이 완주했어도 **Crashed**로
표시합니다 — 통신 두절 기록일 뿐 학습 실패가 아닙니다. 학습 셀 끝에서 `finish()`를 호출해
**Finished**로 마감합니다.

In [ ]:
USE_WANDB = True

if USE_WANDB:
    try:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
        print("wandb: Colab 보안 비밀에서 API 키 로드")
    except Exception:
        import wandb
        wandb.login()  # 키 입력 프롬프트
    os.environ['WANDB_PROJECT'] = 'kormo-lab'

RUN_NAME = f"kormo-1B-{'fw2-' if DATA_SOURCE == 'fineweb2' else ''}{MODE}"

# run ID 관리 — resume이면 저장해둔 ID를 재사용해 끊긴 run에 이어 기록하고,
# 그 외 모드는 global_step이 0부터 시작하므로 새 run 발급
# (같은 run ID에 낮은 스텝을 다시 쓰면 wandb가 로그를 거부함)
if USE_WANDB:
    RUN_ID_FILE = os.path.join(OUTPUT_DIR, 'wandb_run_id.txt')
    if MODE == 'resume' and os.path.exists(RUN_ID_FILE):
        RUN_ID = open(RUN_ID_FILE).read().strip()
        print(f"wandb: 이전 세션 run에 이어서 기록 — {RUN_ID}")
    else:
        RUN_ID = f"{RUN_NAME}-{datetime.now():%Y%m%d-%H%M%S}"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        with open(RUN_ID_FILE, 'w') as f:
            f.write(RUN_ID)
    os.environ['WANDB_RUN_ID'] = RUN_ID
    os.environ['WANDB_RESUME'] = 'allow'   # 같은 ID의 run이 있으면 이어서, 없으면 새로 생성

REPORT_TO = 'wandb' if USE_WANDB else 'none'
print(f"로깅: {REPORT_TO} | run: {RUN_NAME}")

## 5. 학습

데이터 소스별 설정:

| | tutorial | fineweb2 |
|---|---|---|
| epoch | 3 (동일 데이터 ~4 epoch까지 유효) | **1** (전부 신규 데이터) |
| 학습률 | fresh/resume 5e-4, continue 5e-5 | **1e-4 + warmup 500** |
| 평가 주기 | 100 스텝 | 500 스텝 |
| 저장 주기 | 500 스텝 | 1000 스텝 (~17분마다, Drive 쓰기 부하 고려) |

- fineweb2 1B 토큰은 **총 ~61,000 스텝 ≈ 17시간** — Colab 세션이 도중에 끊기는 것이 정상입니다.
  **같은 노트북을 처음부터 다시 실행**하면 resume 모드로 판정되어 마지막 체크포인트에서 이어집니다
  (데이터는 Drive 캐시 로드라 수 분 내에 학습 재개)
- `eval/loss`가 진짜 진행 지표입니다. train loss만 내려가고 eval이 정체하면 암기가 시작된 신호
- 체크포인트는 **optimizer 상태 포함**으로 저장(재개 가능), 최근 1개만 유지
- 학습이 시작되면 셀 출력에 뜨는 wandb run URL을 열어 실시간 곡선을 보세요

In [ ]:
# 데이터 소스별 하이퍼파라미터 분기
if DATA_SOURCE == 'fineweb2':
    # 학습된 가중치 + 대량 신규 데이터: 튜토리얼 continue용 5e-5는 흡수가 너무 느리고,
    # from-scratch용 5e-4는 수렴 가중치를 파괴함 → 중간값 1e-4 + warmup으로 완충
    LEARNING_RATE = 1e-4
    WARMUP_STEPS = 500
    NUM_EPOCHS = 1        # 전부 신규 데이터라 반복 불필요
    EVAL_STEPS = 500      # 총 ~61k 스텝 기준 12분 간격 (100이면 평가 누적 시간이 과도)
    SAVE_STEPS = 1000     # Drive에 ~8GB 쓰기가 수 분 걸리므로 너무 잦으면 오버헤드 큼
else:
    # continue 모드는 이미 수렴한 가중치 위에 학습하므로 from-scratch용 peak LR(5e-4)을
    # 그대로 쓰면 배운 것을 파괴함(train loss 상승) → 관례대로 1/10로 낮춰 이어감
    LEARNING_RATE = 5e-5 if MODE == 'continue' else 5e-4
    WARMUP_STEPS = 0
    NUM_EPOCHS = 3        # 동일 데이터 ~4 epoch까지는 신규 데이터와 유사한 가치 (Muennighoff 2023)
    EVAL_STEPS = 100
    SAVE_STEPS = 500

training_arguments = KORMoTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,  # 기존 체크포인트 보존 (resume에 필요)
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='linear',
    warmup_steps=WARMUP_STEPS,
    logging_steps=10,
    eval_strategy='steps',       # 검증셋 평가 → wandb에 eval/loss·eval/mean_token_accuracy
    eval_steps=EVAL_STEPS,
    per_device_eval_batch_size=1,  # 평가 loss의 fp32 logits 스파이크 축소 (배치4=7.6GiB → 1.9GiB)
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=1,          # 재개용 체크포인트(~8GB) 1개만 유지
    save_only_model=False,       # optimizer 상태 포함 → 정확한 재개 가능
    report_to=REPORT_TO,
    run_name=RUN_NAME,
)

trainer = KORMoTrainer(
    model=model,
    args=training_arguments,
    train_dataset=packed_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorIntraDocMask(tokenizer),
)

# 업스트림 버그 우회: KORMoTrainer가 _metrics를 'train'만 초기화해서
# 첫 평가 시 compute_loss/log의 self._metrics['eval'] 조회가 KeyError로 죽음
from collections import defaultdict
trainer._metrics.setdefault('eval', defaultdict(list))

n_steps = int(len(packed_ds) * NUM_EPOCHS / training_arguments.per_device_train_batch_size)
print(f"모드: {MODE} | LR: {LEARNING_RATE} (warmup {WARMUP_STEPS}) | {NUM_EPOCHS} epoch ≈ {n_steps:,} 스텝")
print(f"검증: {len(val_ds)} 시퀀스 / {EVAL_STEPS} 스텝마다 | 저장: {SAVE_STEPS} 스텝마다")

In [ ]:
if MODE == 'resume':
    trainer.train(resume_from_checkpoint=True)
else:
    trainer.train()

# run을 Finished 상태로 마감 — 호출하지 않으면 학습이 완주했어도
# 나중에 Colab 세션이 죽을 때 heartbeat만 끊겨 "Crashed"로 표기됨
if USE_WANDB:
    import wandb
    wandb.finish()

## 6. 최종 저장 + 생성 테스트

학습이 끝나면 `final/`에 배포용 가중치를 저장합니다. 다음 세션에서는 이 `final/`이 감지되어
자동으로 **continue 모드**(이 가중치 위에 추가 학습)로 시작됩니다.

fresh 1 epoch 직후는 소형 코퍼스라 출력이 옹알이 수준인 게 정상입니다.
loss가 초기 ~11(=ln 125,184, 균등분포)에서 꾸준히 내려왔다면 파이프라인은 정상 동작한 것입니다.

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("최종 모델 저장:", FINAL_DIR)

model.eval()
prompt = "한국의 수도는"
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=True, top_p=0.9)
print(tokenizer.decode(out[0], skip_special_tokens=True))

### (선택) 재개용 체크포인트 정리

`final/` 저장이 끝났으면 재개용 체크포인트(~8GB)는 더 이상 필요 없습니다.
Drive 용량을 아끼려면 아래 셀을 실행하세요. (다음 추가 학습 때 새로 생성됩니다)

In [ ]:
import shutil

for ck in glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')):
    shutil.rmtree(ck)
    print("삭제:", ck)